In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from utils.file_manager import quantDiffusionPath

file_manager = quantDiffusionPath()   # 모든 데이터 경로는 file_manager 로 관리

In [ ]:
def fill_zero_rv_with_sampling(df):
    df_filled = df.copy()
    
    for col in df.columns:
        zero_mask = (df[col] == 0)
        num_zeros = zero_mask.sum()
        
        if num_zeros > 0:
            # 0이 아닌 값들만 추출
            non_zero_values = df.loc[~zero_mask, col].values
            # 0이 아닌 값들 중 랜덤하게 복원 추출
            random_samples = np.random.choice(non_zero_values, size=num_zeros)
            df_filled.loc[zero_mask, col] = random_samples
            
    print("각 자산별 기존 분포를 활용해 0값을 채웠습니다.")
    return df_filled


# 2. 다양한 RV 추정량 계산 함수 정의
def calculate_rvs(O, H, L, C):
    rv_dict = {}
    
    # (1) Parkinson (1980) - High-Low 기반
    # 공식: 1 / (4 * ln2) * (ln(H/L))^2
    rv_dict['Parkinson'] = (1 / (4 * np.log(2))) * (np.log(H / L)**2)
    
    # (2) Garman-Klass (1980) - OHLC 기반
    # 공식: 0.5 * (ln(H/L))^2 - (2*ln2 - 1) * (ln(C/O))^2
    rv_dict['Garman_Klass'] = 0.5 * (np.log(H / L)**2) - (2 * np.log(2) - 1) * (np.log(C / O)**2)
    
    # (3) Rogers-Satchell (1991) - Drift-Independent
    # 공식: ln(H/C)*ln(H/O) + ln(L/C)*ln(L/O)
    rv_dict['Rogers_Satchell'] = np.log(H / C) * np.log(H / O) + np.log(L / C) * np.log(L / O)
    
    # (4) Simple Close-to-Close Volatility (비교용)
    # 5일 이동평균 분산 (일반적으로 쓰이는 방식)
    returns = np.log(C / C.shift(1))
    rv_dict['Close_to_Close'] = returns**2
    
    return rv_dict

In [ ]:
# # # 1. 대상 티커 리스트 정의 (Yahoo Finance 기준)
# tickers = {
#     "DJI": "^DJI",
#     "NASDAQ": "^IXIC",
#     "HSI": "^HSI",
#     "GOLD": "GC=F",
#     "WTI": "CL=F"
# }

# # 2. 데이터 다운로드 (일간, 2025-12-31 까지)
# # group_by='column' 사용 시 Open, High, Low, Close 별로 정렬됨.
# data = yf.download(list(tickers.values()), start='2000-01-01', end='2025-12-31', interval="1d", auto_adjust=False)

# # 원천 OHLCV 는 data/indices/ 폴더에 저장 (경로는 file_manager 로 관리)
# os.makedirs(file_manager.get_data_dir("indices"), exist_ok=True)
# for field, col in [("O", "Open"), ("H", "High"), ("L", "Low"), ("C", "Close"), ("V", "Volume")]:
#     data[col].to_csv(file_manager.get_indices_ohlcv_file(field))

In [ ]:
# JPM (CRSP) OHLCV
df_JPM = pd.read_csv(file_manager.get_jpm_file()).set_index('DlyCalDt').sort_index()
df_JPM = df_JPM[['DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol']]
df_JPM.columns = ['O', 'H', 'L', 'C', 'V']

# 원천 인덱스 OHLCV(data/indices/) + JPM 을 6자산 패널로 조립
data_dict = {}
for field in ["O", "H", "L", "C", "V"]:
    df = pd.read_csv(file_manager.get_indices_ohlcv_file(field), index_col=0)
    df = df[["^DJI", "^IXIC", "^HSI", "GC=F", "CL=F"]]
    df = pd.concat([df, df_JPM[field]], axis=1)
    df.columns = ["DJI", "NASDAQ", "HSI", "GOLD", "WTI_OIL", "JPM"]
    df = df[["DJI", "NASDAQ", "JPM", "HSI", "GOLD", "WTI_OIL"]]
    df = df.dropna()
    df = df.loc["2001-01-02":"2025-12-31", :]
    df.index = pd.to_datetime(df.index)
    data_dict[field] = df

# 조립된 종가 패널 = 새 indices.csv (indices_C + JPM, 2025-12-30 까지)
indices_df = data_dict["C"].copy()
indices_df.index.name = "Date"
indices_df.to_csv(file_manager.get_indices_file())
print(f"saved {file_manager.get_indices_file()}  shape={indices_df.shape}  "
      f"range={indices_df.index.min().date()} ~ {indices_df.index.max().date()}")

In [ ]:
# 1. 기초 데이터 설정 (기존 data_dict 활용)
O = data_dict['O']
H = data_dict['H']
L = data_dict['L']
C = data_dict['C']


# RV 데이터 생성
rv_results = calculate_rvs(O, H, L, C)

In [ ]:
os.makedirs(file_manager.get_data_dir("RV"), exist_ok=True)
for rv_name in rv_results.keys():
    print(rv_name)
    rv_results[rv_name] = fill_zero_rv_with_sampling(rv_results[rv_name])
    rv_results[rv_name].to_csv(file_manager.get_rv_file(rv_name))